### Preprocessing

In [1]:
# Import Library
import pandas as pd
import xarray as xr
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

In [2]:
# open dataset
df = xr.open_dataset("cru_ts4.09.1901.2024.tmp.dat.nc", engine="netcdf4")

# load US states shapefile
us = gpd.read_file("cb_2018_us_state_500k.zip")

# ensure shapefile is lon/lat
if us.crs is None:
    us = us.set_crs(epsg=4326)
us = us.to_crs(epsg=4326)

# remove AK, HI, and territories (keep CONUS)
drop = ["AK", "HI", "PR", "AS", "VI", "GU", "MP"]
us = us[~us["STUSPS"].isin(drop)]

# dissolve to single polygon
us = us.dissolve()

# bounds (+ padding)
minx, miny, maxx, maxy = us.total_bounds
pad = 0.15
minx1, maxx1 = minx - pad, maxx + pad
miny1, maxy1 = miny - pad, maxy + pad

# make xarray lon consistent with -180..180 if needed (CRU sometimes uses 0..360)
if df["lon"].max() > 180:
    df = df.assign_coords(lon=(((df["lon"] + 180) % 360) - 180)).sortby("lon")

# make sure lat is ascending so slice works consistently
df = df.sortby("lat")

# subset to US bounding box
df_us = df.sel(lon=slice(minx1, maxx1), lat=slice(miny1, maxy1))

print(df_us)

<xarray.Dataset> Size: 173MB
Dimensions:  (lon: 116, lat: 50, time: 1488)
Coordinates:
  * lon      (lon) float32 464B -124.8 -124.2 -123.8 ... -68.25 -67.75 -67.25
  * lat      (lat) float32 200B 24.75 25.25 25.75 26.25 ... 48.25 48.75 49.25
  * time     (time) datetime64[ns] 12kB 1901-01-16 1901-02-15 ... 2024-12-16
Data variables:
    tmp      (time, lat, lon) float32 35MB ...
    stn      (time, lat, lon) float64 69MB ...
    mae      (time, lat, lon) float32 35MB ...
    maea     (time, lat, lon) float32 35MB ...
Attributes:
    Conventions:  CF-1.4
    title:        CRU TS4.09 Mean Temperature
    institution:  Data held at British Atmospheric Data Centre, RAL, UK.
    source:       Run ID = 2503051245. Data generated from:tmp.2503051121.dtb
    history:      Wed  5 Mar 13:28:04 GMT 2025 : User f098 : Program makegrid...
    references:   Information on the data is available at http://badc.nerc.ac...
    comment:      Access to these data is available to any registered CEDA user.

In [3]:
tmp = df_us["tmp"]  # (time, lat, lon)

# USA-wide area-weighted mean
us_mean = tmp.mean(("lat", "lon")).rename("us_tmp_mean")
print(us_mean)

<xarray.DataArray 'us_tmp_mean' (time: 1488)> Size: 6kB
array([-0.26785216, -1.5143187 ,  4.3519397 , ..., 14.980808  ,
        7.83977   ,  3.5187068 ], dtype=float32)
Coordinates:
  * time     (time) datetime64[ns] 12kB 1901-01-16 1901-02-15 ... 2024-12-16


In [4]:
# seasonal cycle (climatology) comparison using WMO baseline (1961-1990)
# How far is each month relative to the reference climate? (baseline)
baseline = us_mean.sel(time=slice("1961-01-01", "1990-12-31")) \
                    .groupby("time.month").mean("time")
early = us_mean.sel(time=slice("1901-01-01", "1960-12-31")) \
                 .groupby("time.month").mean("time")

late = us_mean.sel(time=slice("1991-01-01", "2024-12-31")) \
                .groupby("time.month").mean("time")

In [5]:
# Monthly anomalies relative to baseline climatology
anom = us_mean.groupby("time.month") - baseline
anom = anom.rename("us_tmp_anom")

y_anom = anom.values

**Dataset to Use**

In [6]:
# baseline climatology per cell, per month (1961–1990)
tmp_base = tmp.sel(time=slice("1961-01-01", "1990-12-31"))
clim_6190 = tmp_base.groupby("time.month").mean("time")  # dims: (month, lat, lon)

# anomaly per cell, per time
anom_cell = (tmp.groupby("time.month") - clim_6190).rename("anom")

# create one dataset containing both raw data and anomaly
ds_us = xr.Dataset(
    data_vars={
        "tmp": tmp,         # raw USA temps (time, lat, lon)
        "anom": anom_cell        # anomalies (time, lat, lon)
    },
    coords={
        "time": tmp["time"],
        "lat": tmp["lat"],
        "lon": tmp["lon"]
    }
)

print(ds_us)

anom_mean = us_mean.groupby("time.month") - baseline

ds_us["usa_mean_tmp"] = us_mean
ds_us["usa_mean_anom"] = anom_mean.rename("usa_mean_anom")
ds_us = ds_us.sel(time=slice("1901-01", "2024-12")) # using period of 1901-2024 for modelling

ds_us

<xarray.Dataset> Size: 69MB
Dimensions:  (lon: 116, lat: 50, time: 1488)
Coordinates:
  * lon      (lon) float32 464B -124.8 -124.2 -123.8 ... -68.25 -67.75 -67.25
  * lat      (lat) float32 200B 24.75 25.25 25.75 26.25 ... 48.25 48.75 49.25
  * time     (time) datetime64[ns] 12kB 1901-01-16 1901-02-15 ... 2024-12-16
    month    (time) int64 12kB 1 2 3 4 5 6 7 8 9 10 ... 3 4 5 6 7 8 9 10 11 12
Data variables:
    tmp      (time, lat, lon) float32 35MB nan nan nan nan ... -8.2 -8.4 -7.2
    anom     (time, lat, lon) float32 35MB nan nan nan nan ... 2.903 2.903 2.983


<xarray.Dataset> Size: 69MB
Dimensions:        (lon: 116, lat: 50, time: 1488)
Coordinates:
  * lon            (lon) float32 464B -124.8 -124.2 -123.8 ... -67.75 -67.25
  * lat            (lat) float32 200B 24.75 25.25 25.75 ... 48.25 48.75 49.25
  * time           (time) datetime64[ns] 12kB 1901-01-16 ... 2024-12-16
    month          (time) int64 12kB 1 2 3 4 5 6 7 8 9 ... 4 5 6 7 8 9 10 11 12
Data variables:
    tmp            (time, lat, lon) float32 35MB nan nan nan ... -8.2 -8.4 -7.2
    anom           (time, lat, lon) float32 35MB nan nan nan ... 2.903 2.983
    usa_mean_tmp   (time) float32 6kB -0.2679 -1.514 4.352 ... 14.98 7.84 3.519
    usa_mean_anom  (time) float64 12kB 1.316 -2.143 -0.7209 ... 2.72 2.034 3.299

##### Data Preparation for Modelling

In [7]:
# Step 1: Make 2D mesh of coordinates
lon2d, lat2d = np.meshgrid(ds_us.lon.values, ds_us.lat.values)
lon_flat = lon2d.ravel()
lat_flat = lat2d.ravel()

# Step 2: Vectorized GeoDataFrame
pts = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(lon_flat, lat_flat),
    crs="EPSG:4326"
).to_crs("EPSG:3857")  # meters

# Step 3: Load coastline once
coast = gpd.read_file("https://naciscdn.org/naturalearth/50m/physical/ne_50m_coastline.zip")
coast = coast.cx[-130:-60, 15:60].to_crs("EPSG:3857")
coast_union = coast.unary_union

# Step 4: Vectorized distance calculation
pts["dist_coast_m"] = pts.distance(coast_union)

# Step 5: Reshape back to grid
dist_coast_grid = pts["dist_coast_m"].values.reshape(lon2d.shape)

ds_us["distance_to_coast"] = (("lat", "lon"), dist_coast_grid / 1000)  # km

C:\Users\nurma\AppData\Local\Temp\ipykernel_18376\2451517137.py:15: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  coast_union = coast.unary_union


In [ ]:
ds_us.to_netcdf(
    "ds_us (1901-2024).nc",
    format="NETCDF4"
) # save the dataset with distance to coast included